# 08 -- Melon: one hand grows melon instead of wheat

First increment on the `PROGRESS.md` Level 2 item "Per-tile crop choice
by ROI" -- deliberately *not* the ROI logic yet (that needs melon's sell
price behavior, which we haven't explored). This just proves the
multi-crop mechanism works: **hand index 4** (the single-tile,
self-loop zone in `HAND_ZONES`) grows `MELON` instead of `WHEAT`; the
farmer and hands 0-3 are unchanged from `wheat_loop_hands_agent`.

Known going in (from `kaggle_environments`'s `CROPS` constant): `MELON`
costs $80/seed, isn't `ongoing` (one harvest per planting, unlike
`STRAWBERRY`/`TOMATO`), and its `first_yield_day` is 10 (vs. 2 for
`WHEAT`) -- so the hardcoded `crop_age >= 2` harvest check in the
existing agents needs to branch on `tile["crop"]` for this hand.

In [1]:
import sys
from pathlib import Path

from kaggle_environments import make

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kaggriculture_bot.config import CROPS
from kaggriculture_bot.evaluate import evaluate
from kaggriculture_bot.strategies.baseline import (
    DIRECTIONS,
    HAND_ZONES,
    LOOP_ORDER_12,
    LOOP_ORDER_HANDS,
    SPAWN_FEEDER,
    wheat_loop_hands_agent,
)

## Melon economics -- confirm the numbers straight from the engine

Same rule as everywhere else in this project: read it off the installed
package, don't trust a transcription.

In [2]:
CROPS["WHEAT"], CROPS["MELON"]

({'seed': 10,
  'first_yield_day': 2,
  'max_yield_day': 4,
  'interval': 0,
  'max_yield': 6,
  'ongoing': False},
 {'seed': 80,
  'first_yield_day': 10,
  'max_yield_day': 12,
  'interval': 0,
  'max_yield': 6,
  'ongoing': False})

## Building `wheat_loop_melon_agent`

Your turn. Start from `wheat_loop_hands_agent`'s body and change only
what hand index 4 does:

1. **Seed**: buy `MELON` seed separately from the `WHEAT` batch (hand 4
   only ever needs at most 1 in stock -- it has exactly one tile).
2. **Harvest threshold**: hand 4's tile needs `crop_age >= 10`, not `>=
   2` -- but only for hand 4; hands 0-3 and the farmer keep the wheat
   threshold.
3. **Plant**: hand 4 plants `["PLANT", "MELON"]`, not `WHEAT`.
4. **Sell**: add a `SELL MELON` market order when there's melon in the
   shed, alongside the existing `SELL WHEAT` one.

Everything else (loop lookup, `SPAWN_FEEDER`, `HAND_ZONES`, the farmer's
own logic) stays exactly as in `wheat_loop_hands_agent`.

In [3]:
def wheat_loop_melon_agent(obs):
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    melon_seeds = obs["private"]["seeds"].get("MELON", 0)
    market = []
    hands = []

    for i, (hx, hy) in enumerate(obs["farms"][obs["player"]]["hands"]):
        hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **HAND_ZONES[i % len(HAND_ZONES)]}
        target_x, target_y = hand_loop[(hx, hy)]
        if (target_x, target_y) == (hx, hy):
            move = ["PASS"]
        else:
            move = [DIRECTIONS[(target_x - hx, target_y - hy)]]

        if i == 4:
            seeds = melon_seeds
            crop = "MELON"
            threshold = 10
        else:
            seeds = wheat_seeds
            crop = "WHEAT"
            threshold = 2

        hand_tile = obs["farms"][obs["player"]]["tiles"][hy][hx]
        in_own_zone = (hx, hy) in HAND_ZONES[i % len(HAND_ZONES)]
        if hand_tile == "LOCKED" or not in_own_zone:
            op = move
        elif hand_tile is None and seeds > 0:
            op = ["PLANT", crop]
        elif hand_tile is not None and hand_tile.get("kind") == "PLANT":
            crop_age = obs["day"] - hand_tile["planted_day"]
            if crop_age >= threshold:
                op = ["HARVEST"]
            elif not hand_tile.get("watered_today"):
                op = ["WATER"]
            else:
                op = move
        else:
            op = move
        hands.append(op)


    if len(obs["farms"][obs["player"]]["hands"]) < 5:
        market.append(["HIRE"])

    if obs["step"] == 0 or wheat_seeds <= 5:
        market.append(["BUY_SEED", "WHEAT", 6 - wheat_seeds])

    if obs["step"] == 0 or melon_seeds <= 1:
            market.append(["BUY_SEED", "MELON", 2 - melon_seeds])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    melon_in_shed = obs["private"]["shed"].get("MELON", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])
    if melon_in_shed > 0:
            market.append(["SELL", "MELON", 1])

    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]

    return {"farmer": farmer, "hands": hands, "market": market}

In [4]:
test_env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
test_env.run([wheat_loop_melon_agent, "random"])

prev = None
for step in test_env.steps:
    obs = step[0]["observation"]
    tile = obs["farms"][obs["player"]]["tiles"][2][4]
    if tile != prev:
        print(obs["step"], "day", obs["day"], "hour", obs["hour"], "->", tile)
        prev = tile

20 day 0 hour 20 -> {'kind': 'PLANT', 'crop': 'MELON', 'planted_day': 0, 'watered_today': False, 'consecutive_unwatered': 1, 'yield_units': 1, 'max_lifespan_step': 312, 'fertilized_until_day': -1}
21 day 0 hour 21 -> {'kind': 'PLANT', 'crop': 'MELON', 'planted_day': 0, 'watered_today': True, 'consecutive_unwatered': 1, 'yield_units': 1, 'max_lifespan_step': 312, 'fertilized_until_day': -1}
24 day 1 hour 0 -> {'kind': 'PLANT', 'crop': 'MELON', 'planted_day': 0, 'watered_today': False, 'consecutive_unwatered': 0, 'yield_units': 1, 'max_lifespan_step': 312, 'fertilized_until_day': -1}
45 day 1 hour 21 -> {'kind': 'PLANT', 'crop': 'MELON', 'planted_day': 0, 'watered_today': True, 'consecutive_unwatered': 0, 'yield_units': 1, 'max_lifespan_step': 312, 'fertilized_until_day': -1}
48 day 2 hour 0 -> {'kind': 'PLANT', 'crop': 'MELON', 'planted_day': 0, 'watered_today': False, 'consecutive_unwatered': 0, 'yield_units': 1, 'max_lifespan_step': 312, 'fertilized_until_day': -1}
69 day 2 hour 21 ->

### Bug found: hand 4 never actually plants melon

The cell above printed nothing at all the first time it was run --
`tile(4,2)` never changed. Diagnostics (logging every hand's index +
position + `melon_seeds` + money turn by turn) showed hands 0, 2, and 4
permanently glued to the same tile all day, and `melon_seeds` never
dropping below 2 even after 3 full days.

Root cause, confirmed by reading the installed engine source
(`kaggle_environments/envs/kaggriculture/kaggriculture.py`), not
guessed: each turn's `PLANT` actions are applied in a fixed unit order
-- farmer first, then hands by index (0, 1, 2, 3, 4) -- and
`_apply_unit_action` re-checks the tile fresh for each unit
(`kaggriculture.py:367`, `if tile is not None: return`). If two units
share a tile the same turn, only the first (lowest index) to be applied
actually plants; the rest silently no-op. This is separate from the
"not enough seed for N simultaneous PLANT requests" rule in
`AGENTS.md` -- it's a same-tile race, not a seed-supply one.

Hand 4 is always last, and every hand (0-3 too) used to tend *any*
empty tile it merely passed through on the shared `LOOP_ORDER_HANDS`
transit path -- harmless when every hand wants the same crop (as in
`wheat_loop_hands_agent`), but not once one hand wants something
different: hand 4 lost the plant race on every shared tile, every
time, so it never once succeeded.

**Fix**: a hand now only tends (`PLANT`/`WATER`/`HARVEST`) a tile that
is a key in its *own* `HAND_ZONES` entry (`in_own_zone` check below);
everywhere else it just moves through. This also removes hands 0-3's
previous opportunistic tending of shared transit tiles -- the
validation further down compares two changes at once (melon hand +
zone-restricted tending), consistent with this project's "compare
against current best as a whole" convention.

## Validate against the current best

Per this project's rule (recompute the baseline in the same run): the
agent to beat is `wheat_loop_hands_agent`. Run twice (separate
`evaluate()` calls / fresh `env`s), per the 2026-08-19 harness note in
`PROGRESS.md` -- a single call's `n` isn't trustworthy on its own.

Expect a *small* effect: one tile out of ~29 free ones, on a ~10-12 day
cycle within a ~30-day season -- this is about proving the mechanism,
not a big money swing.

In [5]:
print(evaluate(wheat_loop_hands_agent, wheat_loop_melon_agent, n=20))

{'win_rate': 0.0, 'avg_money': 5834.0}


In [6]:
print(evaluate(wheat_loop_hands_agent, wheat_loop_melon_agent, n=20))

{'win_rate': 0.0, 'avg_money': 5814.0}


## Real leaderboard replay: episode 96612755 (our own v4 submission)

Submitted `wheat_loop_melon_agent` (1-tile melon hand) to Kaggle.
Lost, 6698 vs 31477. Before designing anything further, look at what
actually happened on the board -- same rule as every prior replay
analysis in this project (see `PROGRESS.md`, 2026-08-13/2026-08-20).

In [7]:
import json
from collections import Counter

replay = json.load(open(Path.cwd().parent / "replays" / "96612755.json", encoding="utf-8"))
steps = replay["steps"]
ME = 1  # confirmed via replay["info"]["TeamNames"]
print(replay["info"]["TeamNames"], [s["reward"] for s in steps[-1]])

['Barathan Aslan', 'Alejandro Hernández Mairal'] [31477.0, 6698.0]


In [8]:
hand_plant_crop_counter = Counter()
for step in steps:
    action = step[ME].get("action")
    if not isinstance(action, dict):
        continue
    for h in action.get("hands", []):
        if isinstance(h, list) and h and h[0] == "PLANT" and len(h) > 1:
            hand_plant_crop_counter[h[1]] += 1

print("hand PLANT crops across the whole game:", hand_plant_crop_counter)

hand PLANT crops across the whole game: Counter({'WHEAT': 194, 'MELON': 3})


In [9]:
ever_planted = set()
for step in steps:
    obs = step[ME]["observation"]
    tiles = obs["farms"][obs["player"]]["tiles"]
    for y in range(5):
        for x in range(5):
            t = tiles[y][x]
            if isinstance(t, dict) and t.get("kind") == "PLANT":
                ever_planted.add((x, y))

all_nw = {(x, y) for x in range(5) for y in range(5)}
print("NW tiles never planted in 720 turns:", sorted(all_nw - ever_planted))

NW tiles never planted in 720 turns: [(4, 3)]


In [10]:
farmer_visit_counts = Counter()
for step in steps:
    obs = step[ME]["observation"]
    farmer_visit_counts[tuple(obs["farms"][obs["player"]]["farmer"])] += 1

for tile in LOOP_ORDER_12:
    print(tile, farmer_visit_counts.get(tile, 0))

(4, 4) 88
(3, 4) 89
(2, 4) 89
(1, 4) 89
(0, 4) 89
(0, 3) 89
(1, 3) 34
(2, 3) 74
(2, 2) 60
(3, 2) 17
(3, 3) 2
(4, 3) 0


### Finding: `(4,3)` is never once visited by the farmer

Only one NW tile is never planted the whole game: `(4,3)`, right next to
the respawn tile `(4,4)`. The farmer visit counts above show why: early
loop tiles get visited on nearly every one of the 30 days (~89 times
each), but the count drops off fast toward the end of the loop --
`(3,3)` only 2 times, `(4,3)` **zero**. The farmer resets to `(4,4)`
every day and rarely finishes a full 12-tile lap (harvest/water often
push it past the ~24-turn/day budget), so the loop's *last* tiles before
wrapping around are the ones that starve, day after day.

This predates melon -- it is a gap in `LOOP_ORDER_12` itself, created
2026-08-19. It was hidden until now because hands used to opportunistically
tend `(3,3)`/`(4,3)` while passing through on shared `LOOP_ORDER_HANDS`
transit (they are nodes in that graph too) -- the `in_own_zone` fix from
earlier in this notebook removed that accidental coverage and exposed
the real gap.

## Fix: give hand 4 a 2x2 zone instead of 1 tile

`(4,2)` (hand 4's current zone), `(4,3)`, `(3,3)`, and `(3,2)` form a
closed, orthogonally-adjacent 2x2 block -- same shape as zones 0 and 1.
Extending hand 4 into that block plugs the farmer-loop gap (now a real
*own-zone* tile, not opportunistic transit tending) and gives melon 4x
the tile capacity in the same move.

A new constant, not a `HAND_ZONES` mutation -- `wheat_loop_hands_agent`
(already graduated, all-wheat) must stay untouched, same reasoning as
`LOOP_ORDER_2X2` staying separate from `LOOP_ORDER_12`.

In [11]:
MELON_HAND_ZONES = [
    HAND_ZONES[0],
    HAND_ZONES[1],
    HAND_ZONES[2],
    HAND_ZONES[3],
    {(4, 2): (4, 3), (4, 3): (3, 3), (3, 3): (3, 2), (3, 2): (4, 2)},  # hand 4: 2x2 block, was 1 tile
]

`wheat_loop_melon_agent` imported at the top of this notebook is the
1-tile version (what is currently in `src/`, and what episode 96612755
ran). Keep it as the baseline to beat -- swap in `MELON_HAND_ZONES` for
the new version, same body otherwise.

In [12]:
def wheat_loop_melon_zones_agent(obs):
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    melon_seeds = obs["private"]["seeds"].get("MELON", 0)
    market = []
    hands = []

    for i, (hx, hy) in enumerate(obs["farms"][obs["player"]]["hands"]):
        hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **MELON_HAND_ZONES[i % len(MELON_HAND_ZONES)]}
        target_x, target_y = hand_loop[(hx, hy)]
        if (target_x, target_y) == (hx, hy):
            move = ["PASS"]
        else:
            move = [DIRECTIONS[(target_x - hx, target_y - hy)]]

        if i == 4:
            seeds = melon_seeds
            crop = "MELON"
            threshold = 10
        else:
            seeds = wheat_seeds
            crop = "WHEAT"
            threshold = 2

        hand_tile = obs["farms"][obs["player"]]["tiles"][hy][hx]
        in_own_zone = (hx, hy) in MELON_HAND_ZONES[i % len(MELON_HAND_ZONES)]
        if hand_tile == "LOCKED" or not in_own_zone:
            op = move
        elif hand_tile is None and seeds > 0:
            op = ["PLANT", crop]
        elif hand_tile is not None and hand_tile.get("kind") == "PLANT":
            crop_age = obs["day"] - hand_tile["planted_day"]
            if crop_age >= threshold:
                op = ["HARVEST"]
            elif not hand_tile.get("watered_today"):
                op = ["WATER"]
            else:
                op = move
        else:
            op = move
        hands.append(op)

    if len(obs["farms"][obs["player"]]["hands"]) < 5:
        market.append(["HIRE"])

    if obs["step"] == 0 or wheat_seeds <= 5:
        market.append(["BUY_SEED", "WHEAT", 6 - wheat_seeds])

    if obs["step"] == 0 or melon_seeds <= 1:
        market.append(["BUY_SEED", "MELON", 2 - melon_seeds])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    melon_in_shed = obs["private"]["shed"].get("MELON", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])
    if melon_in_shed > 0:
        market.append(["SELL", "MELON", 1])

    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]

    return {"farmer": farmer, "hands": hands, "market": market}

## Validate against the current best

Current best is `wheat_loop_melon_agent` (1-tile melon, imported at the
top). Run twice, per this project's rule.

In [13]:
print(evaluate(wheat_loop_melon_agent, wheat_loop_melon_zones_agent, n=20))

{'win_rate': 0.0, 'avg_money': 8350.0}


In [14]:
print(evaluate(wheat_loop_melon_agent, wheat_loop_melon_zones_agent, n=20))

{'win_rate': 0.0, 'avg_money': 8553.0}
